#### Task 4: What a 99% confidence level means for a cricket app's match-alert feature.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(seed=40)

print("H0: match alerts do not increase engagement; H1: match alerts increase engagement\n")

# ---- Part A: stricter confidence -> stricter evidence requirement ----
rows = []
for conf in (0.95, 0.99):
    alpha = 1 - conf
    rows.append({"Confidence": f"{conf:.0%}", "alpha (Type I risk)": alpha,
                 "Critical z (one-tailed)": stats.norm.ppf(1 - alpha)})
print(pd.DataFrame(rows).round(3).to_string(index=False), "\n")

# ---- Part B: one experiment, two decisions ----
n, sd = 300, 12                     # users per group, SD of daily minutes in app
mean_alerts, mean_control = 32.0, 30.0
t_stat, p_value = stats.ttest_ind_from_stats(mean_alerts, sd, n, mean_control, sd, n,
                                             equal_var=False, alternative="greater")
print(f"Experiment: alerts group {mean_alerts} min vs control {mean_control} min (n = {n} each)")
print(f"t = {t_stat:.3f}, p-value = {p_value:.4f}")
for conf in (0.95, 0.99):
    alpha = 1 - conf
    verdict = "REJECT H0 -> ship the feature" if p_value < alpha else "FAIL TO REJECT H0 -> not proven, do not ship yet"
    print(f"  At {conf:.0%} confidence (alpha = {alpha:.2f}): {verdict}")

# ---- Part C: the price of being stricter (lower power) ----
trials = 4000
pvals = np.array([stats.ttest_ind(rng.normal(mean_alerts, sd, n), rng.normal(mean_control, sd, n),
                                  alternative="greater").pvalue for _ in range(trials)])
print("\nIf alerts truly add 2 minutes, how often does each confidence level detect it?")
power = {}
for conf in (0.95, 0.99):
    power[conf] = np.mean(pvals < 1 - conf)
    print(f"  {conf:.0%} confidence: detected in {power[conf]:.1%} of experiments "
          f"(missed = Type II error {1 - power[conf]:.1%})")

H0: match alerts do not increase engagement; H1: match alerts increase engagement

Confidence  alpha (Type I risk)  Critical z (one-tailed)
       95%                 0.05                    1.645
       99%                 0.01                    2.326 

Experiment: alerts group 32.0 min vs control 30.0 min (n = 300 each)
t = 2.041, p-value = 0.0208
  At 95% confidence (alpha = 0.05): REJECT H0 -> ship the feature
  At 99% confidence (alpha = 0.01): FAIL TO REJECT H0 -> not proven, do not ship yet

If alerts truly add 2 minutes, how often does each confidence level detect it?
  95% confidence: detected in 66.1% of experiments (missed = Type II error 33.9%)
  99% confidence: detected in 39.8% of experiments (missed = Type II error 60.2%)
